<a href="https://colab.research.google.com/github/Mati1491/Tp_final/blob/main/tp_programacion_avanzada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# agrrgamos patrones de diseño y meta clase

from datetime import date

# ==============================
# 1. Metaclase para el Singleton
# ==============================

class SingletonMeta(type):
    """
    Metaclase para que haya una sola instancia de la clase que la use.
    La usamos en Biblioteca para que no se creen dos bibliotecas distintas.
    """
    _instancias = {}

    def __call__(cls, *args, **kwargs):
        # Si todavía no se creó esa clase, la creamos y la guardamos
        if cls not in cls._instancias:
            cls._instancias[cls] = super().__call__(*args, **kwargs)
        # Siempre devolvemos la misma instancia
        return cls._instancias[cls]


# ============================
# 2. Decorador de auditoría
# ============================

def registrar_accion(funcion):
    """
    Decorador simple que muestra un mensaje cada vez que se llama
    a una función importante del sistema (alta, baja, préstamo, etc.).
    Así podemos ver en la consola qué operaciones se están ejecutando.
    """

    def wrapper(*args, **kwargs):
        print(f"[AUDITORÍA] Ojo, se está ejecutando: '{funcion.__name__}'")
        # Acá llamamos a la función original del sistema
        return funcion(*args, **kwargs)

    return wrapper

# ============================
# 3. Clases del modelo de datos
# ============================

class Libro:
    """
    Representa un libro dentro de la biblioteca.
    """
    def __init__(self, titulo, autor, isbn, anio_publicacion, cantidad_paginas):
        self.titulo = titulo
        self.autor = autor
        self.isbn = isbn
        self.anio_publicacion = anio_publicacion
        self.cantidad_paginas = cantidad_paginas

    # Polimorfismo: este método existe en varias clases
    def mostrar_info(self):
        return (f"LIBRO -> Título: {self.titulo} | Autor: {self.autor} | "
                f"ISBN: {self.isbn}")


class Persona:
    """
    Clase base para armar la jerarquía de herencia de usuarios.
    """
    def __init__(self, nombre, apellido, dni):
        self.nombre = nombre
        self.apellido = apellido
        self.dni = dni

    # Versión base del método polimórfico
    def mostrar_info(self):
        return f"Persona: {self.nombre} {self.apellido}"


class Usuario(Persona):
    """
    Usuario de la biblioteca. Hereda de Persona y agrega el email.
    """
    def __init__(self, nombre, apellido, dni, email):
        # Usamos super() para reutilizar el constructor de Persona
        super().__init__(nombre, apellido, dni)
        self.email = email

    # Polimorfismo: sobreescribimos mostrar_info
    def mostrar_info(self):
        return (f"USUARIO -> {self.apellido}, {self.nombre} | "
                f"DNI: {self.dni} | Email: {self.email}")


class Prestamo:
    """
    Clase que maneja los préstamos de libros a usuarios.
    """
    def __init__(self, usuario, libro):
        self.usuario = usuario
        self.libro = libro
        self.fecha_prestamo = date.today()
        self.devuelto = False

    def registrar_devolucion(self):
        """
        Marcamos el préstamo como devuelto.
        (Podríamos guardar también la fecha de devolución).
        """
        self.devuelto = True

    # Versión polimórfica del mostrar_info para préstamos
    def mostrar_info(self):
        estado = "DEVUELTO" if self.devuelto else "ACTIVO"
        return (f"PRÉSTAMO -> Libro: '{self.libro.titulo}' prestado a "
                f"{self.usuario.nombre} el {self.fecha_prestamo} "
                f"({estado})")

# ==========================================
# 4. Clase principal: Sistema de Biblioteca
# ==========================================

class Biblioteca(metaclass=SingletonMeta):
    """
    Clase central del sistema. Acá gestionamos libros, usuarios y préstamos.
    Usamos la metaclase SingletonMeta para que haya una sola Biblioteca.
    """
    def __init__(self):
        # Agregación: la biblioteca tiene listas de usuarios y libros
        self.usuarios = []
        self.libros = []
        # Composición: la biblioteca controla la lista de préstamos
        self.prestamos = []

    # -------------------------
    # Gestión de usuarios
    # -------------------------

    @registrar_accion
    def alta_usuario(self, usuario):
        """
        Damos de alta un usuario nuevo, si el DNI no está repetido.
        """
        for u in self.usuarios:
            if u.dni == usuario.dni:
                print(f"El usuario con DNI {u.dni} ya está ingresado")
                return
        self.usuarios.append(usuario)
        print(f"Usuario {usuario.dni} registrado")

    @registrar_accion
    def baja_usuario(self, dni):
        """
        Eliminamos un usuario por DNI.
        """
        for i, u in enumerate(self.usuarios):
            if u.dni == dni:
                del self.usuarios[i]
                print(f"Usuario con DNI {dni} eliminado")
                return
        print(f"No se encontró un usuario con ese DNI")

    @registrar_accion
    def modificar_usuario(self, dni, nuevo_nombre=None,
                          nuevo_apellido=None, nuevo_email=None):
        """
        Modificamos los datos de un usuario. Solo cambiamos lo que nos pasan.
        """
        for u in self.usuarios:
            if u.dni == dni:
                if nuevo_nombre is not None:
                    u.nombre = nuevo_nombre
                if nuevo_apellido is not None:
                    u.apellido = nuevo_apellido
                if nuevo_email is not None:
                    u.email = nuevo_email
                print("Usuario modificado con éxito")
                return
        print("No se encontró un usuario con ese DNI")

    def buscar_usuario(self, dni):
        """
        Buscamos un usuario por DNI y lo devolvemos, o None si no existe.
        """
        for u in self.usuarios:
            if u.dni == dni:
                return u
        return None

    # -------------------------
    # Gestión de libros
    # -------------------------

    @registrar_accion
    def alta_libro(self, libro):
        """
        Damos de alta un libro nuevo, si el ISBN no está repetido.
        """
        for l in self.libros:
            if l.isbn == libro.isbn:
                print(f"El libro {libro.isbn} ya está registrado")
                return
        self.libros.append(libro)
        print(f"Libro {libro.isbn} ha sido registrado")

    @registrar_accion
    def baja_libro(self, isbn):
        """
        Eliminamos un libro por ISBN.
        """
        for i, l in enumerate(self.libros):
            if l.isbn == isbn:
                del self.libros[i]
                print(f"Libro con ISBN {isbn} eliminado")
                return
        print("No se encontró un libro con ese ISBN")

    @registrar_accion
    def modificar_libro(self, isbn, nuevo_titulo=None,
                        nuevo_autor=None, nuevo_anio=None,
                        nuevas_paginas=None):
        """
        Modificamos los datos de un libro. Solo cambiamos lo que nos pasan.
        """
        for l in self.libros:
            if l.isbn == isbn:
                if nuevo_titulo is not None:
                    l.titulo = nuevo_titulo
                if nuevo_autor is not None:
                    l.autor = nuevo_autor
                if nuevo_anio is not None:
                    l.anio_publicacion = nuevo_anio
                if nuevas_paginas is not None:
                    l.cantidad_paginas = nuevas_paginas
                print("Libro modificado con éxito")
                return
        print("No se encontró un libro con ese ISBN")

    def buscar_libro(self, isbn):
        """
        Buscamos un libro por ISBN y lo devolvemos, o None si no existe.
        """
        for l in self.libros:
            if l.isbn == isbn:
                return l
        return None

    # -------------------------
    # Gestión de préstamos
    # -------------------------

    @registrar_accion
    def registrar_prestamo(self, dni, isbn):
        """
        Registramos un préstamo entre un usuario y un libro.
        No permitimos prestar un libro que ya tiene un préstamo activo.
        """
        usuario = self.buscar_usuario(dni)
        libro = self.buscar_libro(isbn)

        if usuario is None or libro is None:
            print("Error: Usuario o Libro no encontrados")
            return

        # Chequeamos si el libro ya tiene un préstamo activo
        for prestamo in self.prestamos:
            if prestamo.libro.isbn == isbn and not prestamo.devuelto:
                print("Ese libro ya tiene un préstamo activo")
                return

        # Composición en acción: creamos el Prestamo dentro de la biblioteca
        nuevo_prestamo = Prestamo(usuario, libro)
        self.prestamos.append(nuevo_prestamo)
        print("Préstamo registrado correctamente")

    @registrar_accion
    def devolver_libro(self, isbn):
        """
        Registramos la devolución de un libro, si hay un préstamo activo.
        """
        for prestamo in self.prestamos:
            if prestamo.libro.isbn == isbn and not prestamo.devuelto:
                prestamo.registrar_devolucion()
                print("Libro devuelto correctamente")
                return
        print("No existe préstamo activo para ese libro")

    def listar_prestamos_activos(self):
        """
        Mostramos todos los préstamos que todavía están activos.
        """
        print("\n=== PRÉSTAMOS ACTIVOS ===")
        hay_prestamos = False
        for prestamo in self.prestamos:
            if not prestamo.devuelto:
                print(prestamo.mostrar_info())
                hay_prestamos = True
        if not hay_prestamos:
            print("No hay préstamos activos")

    # -------------------------
    # Polimorfismo: fichas
    # -------------------------

    def imprimir_ficha(self, objeto_del_sistema):
        """
        Método polimórfico: recibe cualquier objeto del sistema que tenga
        mostrar_info() (Libro, Usuario o Prestamo) y muestra su ficha.
        """
        print("\n--- FICHA GENERADA ---")
        print(objeto_del_sistema.mostrar_info())


In [ ]:
# probamos que funcion
# 1) Probamos el Singleton (metaclase)
print("=== 1. TESTEANDO EL SINGLETON (LA METACLASE) ===")
b1 = Biblioteca()
b2 = Biblioteca()
print(f"¿b1 y b2 son exactamente la misma biblioteca en memoria?: {b1 is b2}")

# 2) Creamos objetos de prueba (Libro y Usuario)
print("\n=== 2. CREANDO OBJETOS DE PRUEBA ===")
user = Usuario("Carlos", "Sánchez", "35123456", "carlos@unab.edu.ar")
book = Libro("Introducción a Python", "Guido van Rossum", "978-987", 2020, 350)

# 3) Probamos el decorador en altas
print("\n=== 3. PROBANDO EL DECORADOR ===")
b1.alta_usuario(user)
b1.alta_libro(book)

# 4) Probamos composición registrando un préstamo
print("\n=== 4. PROBANDO COMPOSICIÓN ===")
b1.registrar_prestamo("35123456", "978-987")

# 5) Probamos polimorfismo con imprimir_ficha
print("\n=== 5. PROBANDO POLIMORFISMO ===")
# Le mandamos tres cosas distintas al mismo método y todas responden bien
b1.imprimir_ficha(book)           # usa mostrar_info de Libro
b1.imprimir_ficha(user)           # usa mostrar_info de Usuario
b1.imprimir_ficha(b1.prestamos[0])  # usa mostrar_info de Prestamo

# 6) Mostramos préstamos activos
b1.listar_prestamos_activos()

=== 1. TESTEANDO EL SINGLETON (LA METACLASE) ===
¿b1 y b2 son exactamente la misma biblioteca en memoria?: True

=== 2. CREANDO OBJETOS DE PRUEBA ===

=== 3. PROBANDO EL DECORADOR ===
[AUDITORÍA] Ojo, se está ejecutando: 'alta_usuario'
Usuario 35123456 registrado
[AUDITORÍA] Ojo, se está ejecutando: 'alta_libro'
Libro 978-987 ha sido registrado

=== 4. PROBANDO COMPOSICIÓN ===
[AUDITORÍA] Ojo, se está ejecutando: 'registrar_prestamo'
Préstamo registrado correctamente

=== 5. PROBANDO POLIMORFISMO ===

--- FICHA GENERADA ---
LIBRO -> Título: Introducción a Python | Autor: Guido van Rossum | ISBN: 978-987

--- FICHA GENERADA ---
USUARIO -> Sánchez, Carlos | DNI: 35123456 | Email: carlos@unab.edu.ar

--- FICHA GENERADA ---
PRÉSTAMO -> Libro: 'Introducción a Python' prestado a Carlos el 2026-06-28 (ACTIVO)

=== PRÉSTAMOS ACTIVOS ===
PRÉSTAMO -> Libro: 'Introducción a Python' prestado a Carlos el 2026-06-28 (ACTIVO)
